# **EE4C12-Epileptic-Seizure-Detection-using-EEG**

## *Classification of EEG signals using **Logistic regression***


    
Group 16 Members:

    1. Zhixuan Ge  
    2. Yanqi Hong 

## 1. Import Libraries and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import f1_score, accuracy_score, roc_curve, precision_recall_curve, PrecisionRecallDisplay, roc_auc_score, RocCurveDisplay, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE


In [ ]:
random_state = 42

In [ ]:
#load data on Yanqi Hong's computer
data = pd.read_csv('E:\DATA\TUD\Master\TUD_Master_Y1\Q1\EE4C12 Machine Learning For Electrical Engineering\CodeLab\Project\S&S_SZD (1)\Data\Project_Data_EE4C12_S&S_SZD.csv')
data

# # load data on Zhixuan's computer
# data = pd.read_csv('D:\\User\Zhixuan Ge\Onedrive TUDelft\OneDrive - Delft University of Technology\Courses\ML for EE\SZD\S&S_SZD\Project_Data_EE4C12_S&S_SZD.csv')
# data

## 2. Data Preprocessing

In [ ]:
# train test split

X = data.iloc[:, 2:].values
y = np.int32(data['annotation'].values)

test_size = 0.25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

In [ ]:
# standardize data

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Feature selection by extra tree classifier

print("The original number of training feature is: ", X_train_scaled.shape[1])
clf_etc = ExtraTreesClassifier(random_state=random_state).fit(X_train_scaled, y_train) # fit the model
feature_importances = clf_etc.feature_importances_  # get the feature importance

plt.bar(range(len(feature_importances)), feature_importances)   # plot the feature importance
plt.xlabel("Feature index")
plt.ylabel("Feature importance")
plt.title("Feature importance of all features")
plt.show()

important_feature_indices=np.argsort(feature_importances)   # sort the feature importance  
important_feature_indices_cut=important_feature_indices[:int(len(important_feature_indices)/1.5)]   # select the most important features  

X_train_selected=np.delete(X_train_scaled,important_feature_indices_cut,1)    # delete the least important features
X_test_selected=np.delete(X_test_scaled,important_feature_indices_cut,1)      # delete the least important features
print("The current number of training feature after feature selection is: ", X_train_selected.shape[1])

## 3. Model Training

In [ ]:
# Generate the performance value and confusion matrix 

def performance_eval(true, pred, score_display=True, matrix_display=False, result_return=False):
    Accuracy = accuracy_score(true, pred)
    F1 = accuracy_score(true, pred)
    Precision = accuracy_score(true, pred)
    Recall = accuracy_score(true, pred)
    
    if score_display==True:
        print("Accuracy: " + str(Accuracy))
        print("F1 score: " + str(F1))
        print("Recall score: " + str(Recall))
        print("Precision score: " + str(Precision))
        
    if matrix_display==True:
        label = ['Non-seizure', 'Transition','Seizure']
        cm = confusion_matrix(true, pred)
        cm_display = ConfusionMatrixDisplay(cm, display_labels=label).plot()
        plt.show(cm_display)
    
    if result_return:
        return Accuracy, F1, Precision, Recall

In [ ]:
#Logistic regression model construction for all train sets
clf_lr = LogisticRegression(max_iter=1000, random_state=random_state).fit(X_train_selected, y_train)

In [ ]:
# Generate the performance value and confusion matrix on test set

y_pred = clf_lr.predict(X_test_selected)
performance_eval(y_test, y_pred, matrix_display=True)

In [ ]:
# Random search for hyperparameter tuning

n_iter_search = 500 # number of iterations for random search
clf_lr_rs = LogisticRegression(max_iter=500, solver='saga') # define the model

# define the hyperparameter space
C = 1/np.power(10, np.arange(-4, 1, 1e-4))  
param_dist = {"penalty": ['l1', 'l2', 'elasticnet', None],
              "C": C,
              "l1_ratio": np.arange(0, 1, 1e-4)}

# random search
random_search = RandomizedSearchCV(clf_lr_rs, param_distributions=param_dist,
                                   n_iter=n_iter_search, cv=5, verbose=3, n_jobs=8,
                                   random_state=random_state)

random_search.fit(X_train_selected, y_train)

best_params = random_search.best_params_
best_score = random_search.best_score_


In [ ]:
# Generate the performance value and confusion matrix on test set

y_pred = random_search.predict(X_test_selected)
Accuracy, F1, Precision, Recall=performance_eval(y_test, y_pred, matrix_display=True, result_return=True)

In [ ]:
#linear regression summary

print("-----------------Logistic Regression Summary-----------------")
print("Best parameters:", best_params)
print("Best score in cross validation process:", best_score)
print("Accuracy in test set with best parameters: " + str(Accuracy))
print("-------------------------------------------------------------")